# Notice utilisation 
Ce code permet d'effectuer la discrimination des assignations réalisées avec DataAnalysis 
Les colonnes que le fichier d'export des assignations doit contenir sont les suivantes : 
+ Observed Intens = intensité
+ Observed m/z = masse mesurée
+ err ppm = erreur sur la masse 
+ sum formula = formule brute proposée
+ et aussi une colonne pour chaque atome et chaque isotope

si les colonnes demandées n'existent pas alors le code remplira automatiquement avec des 0 : donc CONSERVER les colonnes initialement définies sur le code (C,H,N,O,S,Na,Cl,Mg et isotopes) même si il n'y a aucune assignation avec ces atomes car sinon le code va renvoyer des erreurs 

# Packages et fonctions

In [4]:
import pandas as pd
import glob
import os
import numpy as np

# permet de faire les comparaisons masses-masses pour retirer les contaminations
def tronquer_4_decimales(x):
    try:
        x_str = f"{float(x):.10f}"
        partie_entiere, partie_decimale = x_str.split(".")
        return f"{partie_entiere}.{partie_decimale[:4]}"
    except:
        return None

def tronquer_3_decimales(x):
    try:
        x_str = f"{float(x):.10f}"
        partie_entiere, partie_decimale = x_str.split(".")
        return f"{partie_entiere}.{partie_decimale[:3]}"
    except:
        return None

## Fonctions pour attribuer l'indice de confiance

### Cas des molécules organiques classiques CHO, CHNO, CHNOS...

In [1093]:
# VALIDATION DU SOUFRE 
parametres_soufre = {
    1: {"norm": 95, "isotopes": {1: 4},  "lim": 3.6e7},
    2: {"norm": 90, "isotopes": {1: 8},  "lim": 1.7e7},
    3: {"norm": 86, "isotopes": {1: 11}, "lim": 1.2e7},
    4: {"norm": 82, "isotopes": {1: 14}, "lim": 8.8e6},
    5: {"norm": 77, "isotopes": {1: 17}, "lim": 6.8e6},
    6: {"norm": 74, "isotopes": {1: 20}, "lim": 5.6e6},
    7: {"norm": 70, "isotopes": {1: 22}, "lim": 4.8e6},}

def indice_confiance_CHNOS(df):
    df = df.copy()
    # Boucle sur chaque massif isotopique (groupe)
    group_cols = ['H','O','N','C_tot','S_tot']
    results = []

    # Boucle sur chaque groupe unique de formule brute
    for _, group in df.groupby(group_cols):
        # Vérification de la présence de la formule mère (mono isotopique)
        mere = group[(group["^13C"] == 0) & (group["^34S"] == 0)]
        if mere.empty:
            continue
        intens_mere = mere["Observed Intens"].values[0]

        # Vérification de la présence des isotopes 
        has_13C = (group["^13C"] > 0).any()
        has_34S = (group["^34S"] > 0).any()
        Stot = group["S_tot"].iloc[0]

        if Stot > 0:  # CAS ATTRIBUTION AVEC SOUFRE 
            S_atoms = int(Stot)
            if S_atoms in parametres_soufre:
                val_centrale = parametres_soufre[S_atoms]["isotopes"][1]
                lim = parametres_soufre[S_atoms]["lim"]

                if has_34S: #^34S présent 
                    obs = group.loc[group["^34S"] > 0, "Observed Intens"].values
                    intens_obs = obs[0] 

                    intens_ref = intens_mere / parametres_soufre[S_atoms]["norm"]
                    borne_5  = (intens_ref * (val_centrale - 5), intens_ref * (val_centrale + 5))
                    borne_20 = (intens_ref * (val_centrale - 20), intens_ref * (val_centrale + 20))

                    if borne_5[0] <= intens_obs <= borne_5[1]:
                        idx = 1
                    elif borne_20[0] <= intens_obs <= borne_20[1]:
                        idx = 2
                    elif has_13C:
                        idx = 3
                    else:
                        continue
                else:  # ^34S absent
                    if has_13C:
                        idx = 3
                    elif intens_mere < lim:
                        idx = 4
                    else:
                        continue

        else:  # CAS ATTRIBUTION SANS SOUFRE 
            if has_13C:
                idx = 3
            else:
                idx = 4

        # Appliquer l’indice à toutes les lignes du groupe 
        group = group.copy()
        group["indice de confiance"] = idx
        results.append(group)

    # Concaténer tous les groupes valides
    if not results:
        return pd.DataFrame(columns=df.columns)

    attrib_valide_final = pd.concat(results).reset_index(drop=True)
    return attrib_valide_final

### Cas des molécules avec magnéisum

In [1095]:
# VALIDATION DU MAGNESIUM 
parametres_mg = {
    1: {"norm": 79, "isotopes": {1: 10, 2: 11}},  
}

def id_confiance_magnesium(row):
    isotopes = row['list_Mg']
    intensites = row['intensities']
    params = parametres_mg[1]

    i0 = isotopes.index(0)
    intens_M0 = intensites[i0]
    intens_ref = intens_M0 / params["norm"]

    # Vérification isotopes 25Mg et 26Mg
    check = {}
    for iso, val_centrale in params["isotopes"].items():
        if iso in isotopes:
            i_iso = isotopes.index(iso)
            intens_iso = intensites[i_iso]

            attendu = intens_ref * val_centrale
            borne_5 = (intens_ref * (val_centrale - 5),intens_ref * (val_centrale + 5))
            borne_20 = (intens_ref * (val_centrale - 20),intens_ref * (val_centrale + 20))

            if borne_5[0] <= intens_iso <= borne_5[1]:
                check[iso] = "5%"
            elif borne_20[0] <= intens_iso <= borne_20[1]:
                check[iso] = "20%"
            else:
                check[iso] = "out"
        else:
            check[iso] = "absent"

    # Attribution des niveaux de confiance
    if check[1] == "5%" and check[2] == "5%":
        return 1
    elif check[1] in ("5%", "20%") and check[2] in ("5%", "20%"):
        return 2
    elif check[1] in ("5%", "20%") or check[2] in ("5%", "20%"):
        return 3
    elif check[1] == "absent" and check[2] == "absent" and intens_M0 < 2e7:
        return 4
    else:
        return 0

def indice_confiance_Mg(df):
    df = df.copy()
    # Boucle sur chaque massif isotopique (groupe)
    group_cols = ['H','O','C_tot','Mg_tot','S_tot']
    results = []

    for _, group in df.groupby(group_cols):
        # Vérification de la présence de la formule mère (mono isotopique) 
        mere = group[
            (group['^13C'] == 0) &
            (group['^25Mg'] == 0) &
            (group['^26Mg'] == 0) &
            (group['^34S'] == 0)
        ]
        if mere.empty:
            continue # si pas de formule mère (ie. isotope seul) -> ne pas conserver 

        # Liste des isotopes et intensités
        mg_rows = group[(group['^13C']==0) & (group['^34S']==0)]
        list_Mg = []
        intensities = []
        for _, row in mg_rows.iterrows():
            if row['^25Mg'] > 0:
                iso = 1  # correspond à parametres_mg[1] -> ^25Mg
            elif row['^26Mg'] > 0:
                iso = 2  # correspond à parametres_mg[1] -> ^26Mg
            else:
                iso = 0  # mère 
            list_Mg.append(iso)
            intensities.append(row['Observed Intens'])

        # Calcul de l’indice de confiance
        indice = id_confiance_magnesium(pd.Series({
            'list_Mg': list_Mg,
            'intensities': intensities}))
        if indice == 0:
            continue  # pas de confiance → on rejette le groupe

        # Appliquer l’indice à toutes les lignes du groupe
        group = group.copy()
        group['indice de confiance'] = indice
        results.append(group)

    # Concaténer tous les groupes valides
    if not results:
        return pd.DataFrame(columns=df.columns)

    attrib_valide_final = pd.concat(results).reset_index(drop=True)
    return attrib_valide_final

### Cas des molécules avec chlore

In [1097]:
# VALIDATION DU CHLORE 
parametres_cl = {
    1: {"norm": 76, "isotopes": {1: 24}},
    2: {"norm": 57,  "isotopes": {1: 37, 2: 6}},
    3: {"norm": 44,  "isotopes": {1: 42, 2: 13}},
    4: {"norm": 33,  "isotopes": {1: 42, 2: 20, 3: 4}},
    5: {"norm": 25,  "isotopes": {1: 40, 2: 26, 3: 8}},
    6: {"norm": 19,  "isotopes": {1: 36, 2: 29, 3: 12}},
    7: {"norm": 14,  "isotopes": {1: 32, 2: 31, 3: 16}},
    8: {"norm": 11,  "isotopes": {1: 28, 2: 31, 3: 20}},
    9: {"norm": 8,  "isotopes": {1: 24, 2: 30, 3: 23}},
    10: {"norm": 6,  "isotopes": {1: 20, 2: 29, 3: 24}},
    11: {"norm": 5,  "isotopes": {1: 17, 2: 27, 3: 25}},
    12: {"norm": 4,  "isotopes": {1: 14, 2: 24, 3: 26}}}

def id_confiance_chlore(row):
    cl_tot = row['Cl_tot']
    isotopes = row['list_37Cl']
    intensites = row['intensities']

    params = parametres_cl[cl_tot]

    # Pic de référence 37Cl=0
    if 0 not in isotopes:
        return 0
    i0 = isotopes.index(0)
    intens_M0 = intensites[i0]
    intens_ref = intens_M0 / params["norm"]

    isotopes_valides = set()

    # Vérification des isotopes attendus
    for iso, val_centrale in params["isotopes"].items():
        if iso in isotopes:
            i_iso = isotopes.index(iso)
            intens_iso = intensites[i_iso]

            borne_min = intens_ref * (val_centrale - 5)
            borne_max = intens_ref * (val_centrale + 5)

            if borne_min <= intens_iso <= borne_max:
                isotopes_valides.add(iso)
    
    # Vérification des cas
    if cl_tot == 1:
        return 1 if 1 in isotopes_valides else 0

    elif cl_tot in [2, 3]:
        if {1, 2}.issubset(isotopes_valides):
            return 1
        elif 1 in isotopes_valides:
            return 2
        else:
            return 0

    elif 4 <= cl_tot <= 10:
        if {1, 2, 3}.issubset(isotopes_valides):
            return 1
        elif {1, 2}.issubset(isotopes_valides):
            return 2
        else:
            return 0
    return 0

def indice_confiance_Cl(df):
    df = df.copy()
    # Boucle sur chaque massif isotopique (groupe)
    group_cols = ['H','O','N','S_tot','C_tot','Cl_tot']
    results = []

    for _, group in df.groupby(group_cols):
        # Vérification de la présence de la formule mère (mono isotopique) 
        mere = group[
            (group['^13C'] == 0) &
            (group['^37Cl'] == 0) & 
            (group['^34S'] == 0)]
        if mere.empty:
            continue # si pas de formule mère (ie. isotope seul) -> ne pas conserver 

        # Liste des isotopes et intensités
        cl_rows = group[group['^37Cl'].isin([1,2,3]) | (group['^37Cl'] == 0)]
        list_37Cl = []
        intensities = []
        for _, row in cl_rows.iterrows():
            if row['^37Cl'] == 0 :
                iso = 0 # correspond à l'attribution mère 
            else :
                iso = row['^37Cl']  
            list_37Cl.append(iso)
            intensities.append(row['Observed Intens'])

        # Calcul de l’indice de confiance
        indice_cl = id_confiance_chlore(pd.Series({
            'Cl_tot': group['Cl_tot'].iloc[0],  # même pour tout le groupe
            'list_37Cl': list_37Cl,
            'intensities': intensities}))
        if indice_cl == 0:
            continue  # pas de confiance → rejet du groupe

        # Appliquer l’indice à toutes les lignes du groupe
        group = group.copy()
        group['indice de confiance'] = indice_cl
        results.append(group)

    # Concaténer tous les groupes valides
    if not results:
        return pd.DataFrame(columns=df.columns)

    attrib_valide_final = pd.concat(results).reset_index(drop=True)
    return attrib_valide_final

# 0. Importation des données

In [1099]:
#Import du dossier contenant les fichiers CSV des assignations
dossier_csv = r"C:\Users\arthozoc\Nextcloud\Documents\11. FTICR\2. Traitement des spectres\ESI-\RAW\Renazzo"

#Colonnes à conserver
colonnes_a_conserver = [
"Observed Intens",
"Observed m/z",
"err ppm",
"sum formula",
'C',
'H',
'N',
'O',
'S',
'Mg',
'Cl',
'^13C',
'^34S',
'^25Mg',
'^26Mg',
'^37Cl',
]

# Fusionner tous les fichiers du dossier (dans le cas ou les assignations ont été réalisées par passes et qu'il y a un fichier par famille) 
fichiers = glob.glob(os.path.join(dossier_csv, "*.csv"))
liste_df = [pd.read_csv(fichier, sep=";") for fichier in fichiers]
df_final = pd.concat(liste_df, ignore_index=True, sort=False)

#Ajout automatique des colonnes manquantes
for col in colonnes_a_conserver:
    if col not in df_final.columns:
        df_final[col] = 0

#Colonne à conserver
df_final = df_final[colonnes_a_conserver]
df_final = df_final.fillna(0)

# Calcul du DBE et des totaux avec isotopes 
df_final["C_tot"]  = df_final["C"] + df_final["^13C"]
df_final["S_tot"]  = df_final["S"] + df_final["^34S"]
df_final["Mg_tot"] = df_final["Mg"] + df_final["^25Mg"] + df_final["^26Mg"]
df_final["Cl_tot"] = df_final["Cl"] + df_final["^37Cl"]
df_final["DBE"] = 1+ df_final["C_tot"]+ (df_final["N"]/2) - (df_final["H"]/2) - (df_final["Cl_tot"] / 2)

df_final

,Observed Intens,Observed m/z,err ppm,sum formula,C,H,N,O,S,Mg,...,^13C,^34S,^25Mg,^26Mg,^37Cl,C_tot,S_tot,Mg_tot,Cl_tot,DBE
0,2268762.0,198.113567,0.199,C11 H18 Cl ^13C,11.0,18.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,12.0,0.0,0.0,1.0,3.5
1,2065333.0,222.113580,0.122,C13 H18 Cl ^13C,13.0,18.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,14.0,0.0,0.0,1.0,5.5
2,5876092.0,228.160534,0.101,C13 H24 Cl ^13C,13.0,24.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,14.0,0.0,0.0,1.0,2.5
3,26487328.0,242.176168,0.160,C14 H26 Cl ^13C,14.0,26.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,15.0,0.0,0.0,1.0,2.5
4,2249315.0,254.176202,0.020,C15 H26 Cl ^13C,15.0,26.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,16.0,0.0,0.0,1.0,3.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141272,2211804.0,799.485634,0.221,C39 H82 O6 S4 ^25Mg,39.0,82.0,0.0,6.0,4.0,0.0,...,0.0,0.0,1.0,0.0,0.0,39.0,4.0,1.0,0.0,-1.0
141273,5120992.0,799.557682,-0.050,C48 H85 Mg O S2 ^34S,48.0,85.0,0.0,1.0,2.0,1.0,...,0.0,1.0,0.0,0.0,0.0,48.0,3.0,1.0,0.0,6.5
141274,5120992.0,799.557682,0.110,C43 H86 Mg O4 S3 ^13C,43.0,86.0,0.0,4.0,3.0,1.0,...,1.0,0.0,0.0,0.0,0.0,44.0,3.0,1.0,0.0,2.0
141275,5120992.0,799.557682,0.038,C40 H86 O9 S2 ^25Mg,40.0,86.0,0.0,9.0,2.0,0.0,...,0.0,0.0,1.0,0.0,0.0,40.0,2.0,1.0,0.0,-2.0


# 1. Filtrage des données

In [1101]:
# Calcul des ratios
df_final["H/C"] = df_final["H"] / df_final["C_tot"]
df_final["N/C"] = df_final["N"] / df_final["C_tot"]
df_final["O/C"] = df_final["O"] / df_final["C_tot"]
df_final["S/C"] = df_final["S_tot"] / df_final["C_tot"]

# Filtre global (choisir les ratios souhaités) + vérification du DBE
filtre = (
    (df_final["err ppm"].between(-0.2, 0.2)) 
    & (df_final["Observed m/z"].between(150, 800))
    & (df_final["H/C"].between(0.5, 2.5)) 
    & (df_final["N/C"] <= 1)
    & (df_final["O/C"] <= 1)
    & (df_final["S/C"] <= 0.8)
    & (df_final["DBE"] % 1 == 0.5) #DBE demi entier
    & (df_final["DBE"] >= 0.5)  
)

df_final = df_final[filtre].copy()
df_final

,Observed Intens,Observed m/z,err ppm,sum formula,C,H,N,O,S,Mg,...,^37Cl,C_tot,S_tot,Mg_tot,Cl_tot,DBE,H/C,N/C,O/C,S/C
0,2268762.0,198.113567,0.199,C11 H18 Cl ^13C,11.0,18.0,0.0,0.0,0.0,0.0,...,0.0,12.0,0.0,0.0,1.0,3.5,1.500000,0.0,0.000000,0.000000
1,2065333.0,222.113580,0.122,C13 H18 Cl ^13C,13.0,18.0,0.0,0.0,0.0,0.0,...,0.0,14.0,0.0,0.0,1.0,5.5,1.285714,0.0,0.000000,0.000000
2,5876092.0,228.160534,0.101,C13 H24 Cl ^13C,13.0,24.0,0.0,0.0,0.0,0.0,...,0.0,14.0,0.0,0.0,1.0,2.5,1.714286,0.0,0.000000,0.000000
3,26487328.0,242.176168,0.160,C14 H26 Cl ^13C,14.0,26.0,0.0,0.0,0.0,0.0,...,0.0,15.0,0.0,0.0,1.0,2.5,1.733333,0.0,0.000000,0.000000
4,2249315.0,254.176202,0.020,C15 H26 Cl ^13C,15.0,26.0,0.0,0.0,0.0,0.0,...,0.0,16.0,0.0,0.0,1.0,3.5,1.625000,0.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141260,2785600.0,796.168059,-0.050,C55 H31 O S2 ^25Mg,55.0,31.0,0.0,1.0,2.0,0.0,...,0.0,55.0,2.0,1.0,0.0,40.5,0.563636,0.0,0.018182,0.036364
141263,3061623.0,797.491549,0.035,C42 H77 Mg O8 S2,42.0,77.0,0.0,8.0,2.0,1.0,...,0.0,42.0,2.0,1.0,0.0,4.5,1.833333,0.0,0.190476,0.047619
141269,4838360.0,799.412015,0.106,C44 H69 Mg O5 S2 ^34S,44.0,69.0,0.0,5.0,2.0,1.0,...,0.0,44.0,3.0,1.0,0.0,10.5,1.568182,0.0,0.113636,0.068182
141271,2211804.0,799.485634,0.154,C45 H73 O8 S ^26Mg,45.0,73.0,0.0,8.0,1.0,0.0,...,0.0,45.0,1.0,1.0,0.0,9.5,1.622222,0.0,0.177778,0.022222


# 2. Recherche et suppression des contaminants

### Suppression des contaminants de la liste de masse totale (issue du peak picking) 
Permettra à la fin de la procédure de post-traitement de pouvoir comparer la liste des points assignés à la liste de masse totale et en déduire le nombre de pics non assignés 

In [1104]:
# fichier excel contenant la liste de masse brute issue du peak picking 
fichier_spectre = r'C:\Users\arthozoc\Nextcloud\Documents\11. FTICR\2. Traitement des spectres\ESI-\Renazzo.xlsx'
mass_liste_brute = pd.read_excel(fichier_spectre,sheet_name='mass_list_brute')  
# fichier UWPR contenant la liste des contaminants usuels (Keller+ 2008) 
df_contaminants = pd.read_excel(r'C:\Users\arthozoc\Nextcloud\Documents\11. FTICR\2. Traitement des spectres\UWPR_CommonMassSpecContaminants.xls',sheet_name='Negative')  
# fichier contenant la liste des pics intenses du blanc (I > 1%) 
df_blanc = pd.read_excel(r'C:\Users\arthozoc\Nextcloud\Documents\11. FTICR\2. Traitement des spectres\ESI-\BLANC\0. BLANC.xlsx')

# Contaminants usuels (4 décimales)
df_contaminants["masse_tronquee"] = (df_contaminants["Mass"].apply(tronquer_4_decimales))
mass_liste_brute["masse_tronquee_4"] = (mass_liste_brute["m/z"].apply(tronquer_4_decimales))
masses_contaminants = (set(df_contaminants["masse_tronquee"])&set(mass_liste_brute["masse_tronquee_4"]))
liste_contaminants_usuels = (df_contaminants[df_contaminants["masse_tronquee"].isin(masses_contaminants)][["Mass","Ion type","Formula for M or subunit or sequence","Possible origin and other comments"]])


# Contaminants du blanc d'extraction (3 décimales)
df_blanc["masse_tronquee"] = (df_blanc["m/z"].apply(tronquer_3_decimales))
mass_liste_brute["masse_tronquee_3"] = (mass_liste_brute["m/z"].apply(tronquer_3_decimales))
masses_blanc = (set(df_blanc["masse_tronquee"])&set(mass_liste_brute["masse_tronquee_3"]))
liste_blanc = (mass_liste_brute[mass_liste_brute["masse_tronquee_3"].isin(masses_blanc)][["m/z", "I"]])

# Suppression des contaminants trouvés 
liste_masse_filtrée = mass_liste_brute[
    (~mass_liste_brute["masse_tronquee_4"].isin(masses_contaminants))
    &
    (~mass_liste_brute["masse_tronquee_3"].isin(masses_blanc))
].copy()

# Suppression des colonnes inutiles 
colonnes_a_supprimer = ["masse_tronquee_4","masse_tronquee_3"]
liste_masse_filtrée.drop(columns=[c for c in colonnes_a_supprimer if c in liste_masse_filtrée.columns],inplace=True)

# Ajout des feuilles dans le fichier excel

with pd.ExcelWriter(
    fichier_spectre,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:

    liste_contaminants_usuels.to_excel(
        writer,
        sheet_name="Liste_contaminants_usuels",
        index=False)

    liste_blanc.to_excel(
        writer,
        sheet_name="Liste_contaminants_blanc",
        index=False)

    liste_masse_filtrée.to_excel(
        writer,
        sheet_name="Liste_masse_finale",
        index=False)

print("Traitement terminé.")

Traitement terminé.


### Suppression des contaminants de la liste des assignations 

In [1106]:
# On conserve uniquement les masses qui sont présentent dans la liste de masse finale (sans les contaminants) 
masses_conservees = set(liste_masse_filtrée["m/z"].round(5)) # la liste de masse issue du peak picking contient 5 décimales, il faut arrondir
df_final = df_final[df_final["Observed m/z"].round(5).isin(masses_conservees)].copy()

df_final

,Observed Intens,Observed m/z,err ppm,sum formula,C,H,N,O,S,Mg,...,^37Cl,C_tot,S_tot,Mg_tot,Cl_tot,DBE,H/C,N/C,O/C,S/C
0,2268762.0,198.113567,0.199,C11 H18 Cl ^13C,11.0,18.0,0.0,0.0,0.0,0.0,...,0.0,12.0,0.0,0.0,1.0,3.5,1.500000,0.0,0.000000,0.000000
1,2065333.0,222.113580,0.122,C13 H18 Cl ^13C,13.0,18.0,0.0,0.0,0.0,0.0,...,0.0,14.0,0.0,0.0,1.0,5.5,1.285714,0.0,0.000000,0.000000
4,2249315.0,254.176202,0.020,C15 H26 Cl ^13C,15.0,26.0,0.0,0.0,0.0,0.0,...,0.0,16.0,0.0,0.0,1.0,3.5,1.625000,0.0,0.000000,0.000000
5,3786756.0,264.160509,0.183,C16 H24 Cl ^13C,16.0,24.0,0.0,0.0,0.0,0.0,...,0.0,17.0,0.0,0.0,1.0,5.5,1.411765,0.0,0.000000,0.000000
6,2212013.0,276.160539,0.065,C17 H24 Cl ^13C,17.0,24.0,0.0,0.0,0.0,0.0,...,0.0,18.0,0.0,0.0,1.0,6.5,1.333333,0.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141254,2919212.0,795.583953,-0.121,C46 H89 Mg O2 S2 ^34S,46.0,89.0,0.0,2.0,2.0,1.0,...,0.0,46.0,3.0,1.0,0.0,2.5,1.934783,0.0,0.043478,0.065217
141260,2785600.0,796.168059,-0.050,C55 H31 O S2 ^25Mg,55.0,31.0,0.0,1.0,2.0,0.0,...,0.0,55.0,2.0,1.0,0.0,40.5,0.563636,0.0,0.018182,0.036364
141263,3061623.0,797.491549,0.035,C42 H77 Mg O8 S2,42.0,77.0,0.0,8.0,2.0,1.0,...,0.0,42.0,2.0,1.0,0.0,4.5,1.833333,0.0,0.190476,0.047619
141271,2211804.0,799.485634,0.154,C45 H73 O8 S ^26Mg,45.0,73.0,0.0,8.0,1.0,0.0,...,0.0,45.0,1.0,1.0,0.0,9.5,1.622222,0.0,0.177778,0.022222


# 3. Discrimination des assignations

## Attribution d'un indice de confiance

In [1109]:
# Séparer les assignations en 3 catégories pour attribuer un indice de confiance en fonction des critères de chaque catégories : chlore, magénium ou classique
def def_categorie(df):
    df = df.copy()
    df["categorie"] = np.where(df["Mg_tot"] > 0,"Mg",np.where(df["Cl_tot"] > 0, "Cl", "CHNOS"))
    return df

def séparer_categories(df):
    return {
        "CHNOS": df[df["categorie"] == "CHNOS"].copy(),
        "Mg": df[df["categorie"] == "Mg"].copy(),
        "Cl": df[df["categorie"] == "Cl"].copy()
    }

def attribution_indice_confiance(df):

    df = def_categorie(df)
    petit_df = séparer_categories(df)

    df_chnos = indice_confiance_CHNOS(petit_df["CHNOS"])
    df_mg    = indice_confiance_Mg(petit_df["Mg"])
    df_cl    = indice_confiance_Cl(petit_df["Cl"])

    df_fusion = pd.concat([df_chnos, df_mg, df_cl], ignore_index=True)

    colonnes_a_supprimer = ["categorie"]

    df_final = df_fusion.drop(columns=[c for c in colonnes_a_supprimer if c in df_fusion.columns])

    return df_final
    
df_final = attribution_indice_confiance(df_final)

df_final

,Observed Intens,Observed m/z,err ppm,sum formula,C,H,N,O,S,Mg,...,C_tot,S_tot,Mg_tot,Cl_tot,DBE,H/C,N/C,O/C,S/C,indice de confiance
0,1952298.0,155.003327,-0.136,C4 H3 N4 O S,4.0,3.0,4.0,1.0,1.0,0.0,...,4.0,1.0,0.0,0.0,5.5,0.750000,1.00000,0.250000,0.250000,4
1,2096474.0,214.966424,0.131,C8 H7 O S3,8.0,7.0,0.0,1.0,3.0,0.0,...,8.0,3.0,0.0,0.0,5.5,0.875000,0.00000,0.125000,0.375000,4
2,16173863.0,171.012144,-0.029,C7 H7 O3 S,7.0,7.0,0.0,3.0,1.0,0.0,...,7.0,1.0,0.0,0.0,4.5,1.000000,0.00000,0.428571,0.142857,4
3,6323844.0,202.984212,-0.008,C7 H7 O3 S2,7.0,7.0,0.0,3.0,2.0,0.0,...,7.0,2.0,0.0,0.0,4.5,1.000000,0.00000,0.428571,0.285714,4
4,3743333.0,151.040067,0.005,C8 H7 O3,8.0,7.0,0.0,3.0,0.0,0.0,...,8.0,0.0,0.0,0.0,5.5,0.875000,0.00000,0.375000,0.000000,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8479,88592192.0,660.542029,0.004,C38 H76 Cl O5 ^13C,38.0,76.0,0.0,5.0,0.0,0.0,...,39.0,0.0,0.0,1.0,1.5,1.948718,0.00000,0.128205,0.000000,1
8480,67129408.0,661.535721,0.009,C39 H76 O5 ^37Cl,39.0,76.0,0.0,5.0,0.0,0.0,...,39.0,0.0,0.0,1.0,1.5,1.948718,0.00000,0.128205,0.000000,1
8481,27595062.0,662.539050,0.047,C38 H76 O5 ^13C ^37Cl,38.0,76.0,0.0,5.0,0.0,0.0,...,39.0,0.0,0.0,1.0,1.5,1.948718,0.00000,0.128205,0.000000,1
8482,6568504.0,700.601519,0.132,C42 H83 Cl N O4,42.0,83.0,1.0,4.0,0.0,0.0,...,42.0,0.0,0.0,1.0,1.5,1.976190,0.02381,0.095238,0.000000,1


In [1110]:
# création d'un code de famille pour récupérer tous les pics d'un massif isotopique
df_final["famille"] = (
    df_final["C_tot"].astype(str) + "_" +
    df_final["H"].astype(str)     + "_" +
    df_final["O"].astype(str)     + "_" +
    df_final["N"].astype(str)     + "_" +
    df_final["S_tot"].astype(str) + "_" +
    df_final["Mg_tot"].astype(str)+ "_" +
    df_final["Cl_tot"].astype(str)+"_"
)

## A) Utilisation des indices de confiance

In [1112]:
# Premier filtrage 
attributions_validees = pd.DataFrame(columns=df_final.columns) # initialisation de la liste des assignations validées
attributions_multiples = pd.DataFrame(columns=df_final.columns) # initialisation de la liste des mutli-assignations
masses_ignorees = set() # création liste des masses ignorées une fois traitées 

for masse in sorted(df_final["Observed m/z"].unique()): # parcours la liste de masse de manière croissante
    if masse in masses_ignorees : 
        continue 

    # Première étape : pour chaque masse on conserve UNIQUEMENT les points avec le meilleur indice de confiance (le plus petit) 
    df_masse = df_final[df_final["Observed m/z"] == masse] # récupération de toutes les assignations pour une masse donnée
    best_confiance = df_masse["indice de confiance"].min() # parmi les assignations identifier le meilleur indice de confiance (le plus petit)
    liste_best = df_masse[df_masse["indice de confiance"] == best_confiance] # conserver uniquement les assignations avec le meilleur indice de confiance

    if len(liste_best) == 1:
        # une seule assignation est proposée au meilleur niveau de confiance = assigation validée
        selected = liste_best.iloc[0]
        famille = selected["famille"]
        famille_rows = df_final[
            (df_final["famille"] == famille) &
            (~df_final["Observed m/z"].isin(masses_ignorees))] # récupérer toute la famille (avec isotopes)
        attributions_validees = pd.concat([attributions_validees, famille_rows])
        masses_ignorees |= set(famille_rows["Observed m/z"]) # supprimer les masses validées de la liste de masse à traiter 
    else:
        # plusieurs assignations sont proposées au meilleur niveau de confiance = multi-assignations
        familles = liste_best["famille"].unique()
        famille_rows = df_final[
            (df_final["famille"].isin(familles)) &
            (~df_final["Observed m/z"].isin(masses_ignorees))] # récupérer toute la famille (avec isotopes)
        attributions_multiples = pd.concat([attributions_multiples, famille_rows])
        masses_ignorees |= set(famille_rows["Observed m/z"]) # supprimer les masses de la liste de masse à traiter 

C:\Users\arthozoc\AppData\Local\Temp\ipykernel_26136\3715159674.py:22: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  attributions_validees = pd.concat([attributions_validees, famille_rows])
C:\Users\arthozoc\AppData\Local\Temp\ipykernel_26136\3715159674.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  attributions_multiples = pd.concat([attributions_multiples, famille_rows])


## B) Restriction des ratios atomiques 

In [1114]:
# Filtrage des multi_attributions pour ne conserver que celles avec un ratio cohérent
attributions_multiples["ALL/C"] = (attributions_multiples["N"]+attributions_multiples["O"]+attributions_multiples["S_tot"]) / attributions_multiples["C_tot"]

filtre = (
    ((attributions_multiples["N/C"] <= 0.4)
    & (attributions_multiples["O/C"] <= 0.8)
    & (attributions_multiples["S/C"] <= 0.4)
    & (attributions_multiples["ALL/C"] <= 1)))

attributions_multiples = attributions_multiples[filtre].copy()
colonnes_a_supprimer = ["ALL/C"]
attributions_multiples = attributions_multiples.drop(columns=[c for c in colonnes_a_supprimer if c in attributions_multiples.columns])

In [1115]:
attributions_multiples_B = pd.DataFrame(columns=attributions_multiples.columns) # nouvelle liste des attributions multiples à l'issue de ce filtrage 
masses_ignorees = set()

# Parcourir de nouveau la liste de masse et ne conserver que si une seule proposition par masse 
for masse in sorted(attributions_multiples["Observed m/z"].unique()): # parcours la liste de masse de manière croissante
    if masse in masses_ignorees : 
        continue
        
    # récupération de toutes les assignations pour une masse donnée
    df_masse = attributions_multiples[attributions_multiples["Observed m/z"] == masse] 
   
    if len(df_masse) == 1:
        # une seule assignation est proposée après filtrage = assigation validée
        selected = df_masse.iloc[0]
        famille = selected["famille"]
        famille_rows = attributions_multiples[
            (attributions_multiples["famille"] == famille) &
            (~attributions_multiples["Observed m/z"].isin(masses_ignorees))] # récupérer toute la famille (avec isotopes)
        attributions_validees = pd.concat([attributions_validees, famille_rows])
        masses_ignorees |= set(famille_rows["Observed m/z"]) # supprimer les masses validées de la liste de masse à traiter 
    else:
        # plusieurs assignations sont proposées après filtrage = multi-asssingations
        familles = df_masse["famille"].unique()
        famille_rows = attributions_multiples[
            (attributions_multiples["famille"].isin(familles)) &
            (~attributions_multiples["Observed m/z"].isin(masses_ignorees))] # récupérer toute la famille (avec isotopes)
        attributions_multiples_B = pd.concat([attributions_multiples_B, famille_rows])
        masses_ignorees |= set(famille_rows["Observed m/z"]) # supprimer les masses de la liste de masse à traiter 

C:\Users\arthozoc\AppData\Local\Temp\ipykernel_26136\3122485576.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  attributions_multiples_B = pd.concat([attributions_multiples_B, famille_rows])


## C) Recherche des familles en CH2

In [1117]:
# Construire la clé famille_ch2 pour les 2 listes 
attributions_multiples_B["famille_ch2"] = (
    attributions_multiples_B["N"].astype(str) + "_" +
    attributions_multiples_B["O"].astype(str) + "_" +
    attributions_multiples_B["S_tot"].astype(str) + "_" +
    attributions_multiples_B["Mg_tot"].astype(str) + "_" +
    attributions_multiples_B["Cl_tot"].astype(str) + "_" +
    attributions_multiples_B["DBE"].astype(str))

attributions_validees["famille_ch2"] = (
    attributions_validees["N"].astype(str) + "_" +
    attributions_validees["O"].astype(str) + "_" +
    attributions_validees["S_tot"].astype(str) + "_" +
    attributions_validees["Mg_tot"].astype(str) + "_" +
    attributions_validees["Cl_tot"].astype(str) + "_" +
    attributions_validees["DBE"].astype(str))

# Chercher pour chaque attribution de la liste attributions_multiples_B si sa famille en CH2 est déjà présente dans attributions_validees 
familles_CH2_attributions_validees = set(attributions_validees["famille_ch2"])

attributions_multiples_B["famille_CH2_validee"] = (attributions_multiples_B["famille_ch2"].isin(familles_CH2_attributions_validees).map({True: "OUI", False: "NON"}))

In [1118]:
attributions_multiples_C = pd.DataFrame(columns=attributions_multiples_B.columns) # nouvelle liste des attributions multiples à l'issue de ce filtrage 
masses_ignorees = set()

# Parcourir de nouveau la liste de masse et ne conserver que si une seule proposition par masse 
for masse in sorted(attributions_multiples_B["Observed m/z"].unique()): # parcours la liste de masse de manière croissante
    if masse in masses_ignorees : 
        continue 
    # récupération de toutes les assignations pour une masse donnée
    df_masse = attributions_multiples_B[attributions_multiples_B["Observed m/z"] == masse] 
    # combien d'assignation ont déjà leur famille CH2 validée 
    df_oui = df_masse[df_masse["famille_CH2_validee"]=="OUI"]
    if len(df_oui) == 1:
        # une seule assignation a une famille en CH2 deja validée 
        selected = df_oui.iloc[0]
        famille = selected["famille"]
        famille_rows = attributions_multiples_B[
            (attributions_multiples_B["famille"] == famille) &
            (~attributions_multiples_B["Observed m/z"].isin(masses_ignorees))] # récupérer toute la famille (avec isotopes)
        attributions_validees = pd.concat([attributions_validees, famille_rows])
        masses_ignorees |= set(famille_rows["Observed m/z"]) # supprimer les masses validées de la liste de masse à traiter 
    else:
        # plusieurs assignations ont une famille en CH2 deja validée 
        familles = df_masse["famille"].unique()
        famille_rows = attributions_multiples_B[
            (attributions_multiples_B["famille"].isin(familles)) &
            (~attributions_multiples_B["Observed m/z"].isin(masses_ignorees))] # récupérer toute la famille (avec isotopes)
        attributions_multiples_C = pd.concat([attributions_multiples_C, famille_rows])
        masses_ignorees |= set(famille_rows["Observed m/z"]) # supprimer les masses de la liste de masse à traiter 

C:\Users\arthozoc\AppData\Local\Temp\ipykernel_26136\342984146.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  attributions_multiples_C = pd.concat([attributions_multiples_C, famille_rows])


## D) Sauvegarde des deux listes : multi-attributions et attributions validées + tableau récap 

In [1120]:
colonnes_a_supprimer = ["C_tot","S_tot","Cl_tot","Mg_tot","indice de confiance", "famille", "famille_ch2","famille_CH2_validee"] # choix des colonnes à ne pas enregistrer 
attributions_multiples_finales = attributions_multiples_C.drop(columns=[c for c in colonnes_a_supprimer if c in attributions_multiples_C.columns])
attributions_validees_finales = attributions_validees.drop(columns=[c for c in colonnes_a_supprimer if c in attributions_validees.columns])

In [1121]:
nb_total = len(liste_masse_filtrée) # liste de masse totale issue du peak picking (sans les contaminants)
# calcul du nombre de pics attribués - multiattribués - non attribués 
nb_attribuees = attributions_validees_finales["Observed m/z"].nunique()
nb_multi = attributions_multiples_finales["Observed m/z"].nunique()
nb_non_attribuees = nb_total - nb_attribuees - nb_multi
# calcul du pourcentage 
pourcentage_attribuees = (nb_attribuees * 100) / nb_total
pourcentage_multi = (nb_multi * 100) / nb_total
pourcentage_non_attribuees = (nb_non_attribuees * 100) / nb_total

df_resume = pd.DataFrame({
    "Catégorie": [
        "Masses attribuées",
        "Masses multi-attribuées",
        "Masses non attribuées"],
    "Nombre": [
        nb_attribuees,
        nb_multi,
        nb_non_attribuees],
    "% du total": [
        pourcentage_attribuees,
        pourcentage_multi,
        pourcentage_non_attribuees]})

with pd.ExcelWriter(
    fichier_spectre,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    
    attributions_validees_finales.to_excel(
        writer,
        sheet_name="Attributions_validées",
        index=False)

    attributions_multiples_finales.to_excel(
        writer,
        sheet_name="Multi_attributions",
        index=False)

    df_resume.to_excel(
        writer,
        sheet_name="Résumé",
        index=False)

# 4. Passage en formules moléculaires 

## A) Massif isotopique - conserver uniquement les molécules mères

In [1124]:
def construire_formules_meres(df):
    # Intensité totale de chaque famille (mère + isotopes)
    intensites = (df.groupby("famille")["Observed Intens"].sum().rename("Intensite_totale"))

    # Conserver uniquement les formules mères
    meres = df[
        (df["^13C"] == 0) &
        (df["^34S"] == 0) &
        (df["^25Mg"] == 0) &
        (df["^26Mg"] == 0) &
        (df["^37Cl"] == 0)
    ].copy()

    # Ajouter une nouvelle colonne intensité totale
    meres = meres.merge(
        intensites,
        left_on="famille",
        right_index=True,
        how="left")

    return meres

attributions_validees_meres = construire_formules_meres(attributions_validees)
attributions_multiples_meres = construire_formules_meres(attributions_multiples_C)

In [1125]:
colonnes_a_supprimer = ["C_tot","S_tot","Cl_tot","Mg_tot","^13C","^25Mg","^26Mg","^34S","^37Cl","indice de confiance", "famille", "famille_ch2","famille_CH2_validee"] # choix des colonnes à ne pas enregistrer 
attributions_multiples_meres = attributions_multiples_meres.drop(columns=[c for c in colonnes_a_supprimer if c in attributions_multiples_meres.columns])
attributions_validees_meres = attributions_validees_meres.drop(columns=[c for c in colonnes_a_supprimer if c in attributions_validees_meres.columns])

## B) Adduits d'ionisation - passage des formules ioniques aux fomrules moléculaires

In [1127]:
def ionisation(row, df):

    if row['Cl'] == 0:
        return 'H'

    match_H = df[
        (df['C'] == row['C']) &
        (df['H'] == row['H'] - 1) &
        (df['O'] == row['O']) &
        (df['N'] == row['N']) &
        (df['S'] == row['S']) &
        (df['Cl'] == row['Cl'] - 1)
    ]

    if not match_H.empty:
        return 'Cl'

    match_Cl = df[
        (df['C'] == row['C']) &
        (df['H'] == row['H'] + 1) &
        (df['O'] == row['O']) &
        (df['N'] == row['N']) &
        (df['S'] == row['S']) &
        (df['Cl'] == row['Cl'] + 1)
    ]

    if not match_Cl.empty:
        return 'H'

    return 'indéfini'

In [1128]:
def trouver_adduit(df):

    df = df.copy()
    df["adduit"] = df.apply(lambda row: ionisation(row, df),axis=1)

    mask_H = df["adduit"] == "H"
    mask_Cl = df["adduit"] == "Cl"
    mask_indefini = df["adduit"] == "indéfini"

    # [M-H]
    df.loc[mask_H, "H"] += 1
    df.loc[mask_H, "DBE"] -= 0.5

    # [M+Cl]
    df.loc[mask_Cl, "Cl"] -= 1
    df.loc[mask_Cl, "DBE"] += 0.5

    # non résolu
    df = df[df["adduit"] != "indéfini"].copy()
    
    return df

In [1129]:
validees_adduits = trouver_adduit(attributions_validees_meres)
multi_adduits = trouver_adduit(attributions_multiples_meres)

validees_adduits = validees_adduits.drop(columns="adduit")
multi_adduits = multi_adduits.drop(columns="adduit")

## C) Sauvegarde des deux listes : multi-attributions et attributions validées : formules moléculaires

In [1131]:
with pd.ExcelWriter(
    fichier_spectre,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:
    
    validees_adduits.to_excel(
        writer,
        sheet_name="Formules_moléculaires_validées",
        index=False)

    multi_adduits.to_excel(
        writer,
        sheet_name="Formules_moléculaires_multi",
        index=False)

In [1132]:
# petit klaxon pour annoncer que le traitement est terminé
from IPython.display import Audio

samples = np.sin(2 * np.pi * 440 * np.linspace(0, 1, 44100))
display(Audio(samples, rate=44100, autoplay=True))